# 6. Feed-forward reconvergence and `Buffer`

A **feed-forward** network contains a fast and slow path that stem from the same source and reconverge at a later point. When a multi-input layer is the sink of the two paths, the FIFO connected to the fast path may fill up before the slow path has enough packets to produce an output. This causes a **backpressure deadlock**.

Elementwise joins such as `Add`, `Multiply`, `Sub`, `Compare`, `Max`,
`Bitwise`, and `Select` consume **packet `i` of every input together**. If one
branch is much deeper than the other, the fast input arrives long before the
slow one. The join must hold the early packets. If it cannot, it drops
`tready`, which stalls the shared fork (fan-out ANDs every consumer's ready),
and the slow branch never gets the remaining data it needs — progress stops.

```
                 +-- Activation -- Activation -- ... --+   <- slow branch
     din --------+                                     +-- Add --> dout
                 +-------------- Buffer ---------------+   <- hold packets on the fast side
```

Hand-built graphs must insert `Buffer` on the **fast** edge explicitly. 


In [1]:
from pathlib import Path

import numpy as np
from fpga_ai_suite.spatial.flow import run_flow
from fpga_ai_suite.spatial.layers import Accumulator, Add, Buffer, Directed_acyclic_graph, Repeater, Sequential

WORK = Path("work_feed_forward_buffers")
WORK.mkdir(exist_ok=True)


## Anti-pattern: reconverge without a `Buffer`

Emulation still succeeds (the Python model does not enforce RTL FIFO depth),
but RTL simulation of unbalanced joins without enough holding can hang. The
topology below is the diamond you should not ship as-is for deep imbalance.


In [2]:
class Reconverge_no_buffer(Directed_acyclic_graph):
    """Slow Activation chain joins a near-wire path at Add — no holding on the fast side."""

    def __init__(self, depth=6, num_lanes=4, dw=16):
        self.depth = depth
        self.num_lanes = num_lanes
        self.dw = dw
        super().__init__(module_name="Reconverge_no_buffer")

    def init_topology(self):
        self.input = self.create_input("din", self.dw, self.num_lanes, "BF16")
        self.output = self.create_output("dout", self.dw, self.num_lanes, "BF16")

        self.slow = self.add_node(
            Sequential(
                name="Slow_path",
                layers=[
                    Accumulator(num_packets=100, dw_in=self.dw, num_lanes=self.num_lanes, dtype="BF16"),
                    Repeater(repeat=100),
                ]
            )
        )

        self.join = self.add_node(Add(num_lanes=self.num_lanes, dw=self.dw, dtype="BF16"))

        # Fan-out: same din into the slow chain and directly into the join
        self.connect(self.input, self.slow.inputs.din)
        self.connect(self.input, self.join.inputs.dinb)
        self.connect(self.slow.outputs.dout, self.join.inputs.dina)
        self.connect(self.join.outputs.dout, self.output)


bad = Reconverge_no_buffer()
data = (np.random.rand(100, 1, 8).astype(np.float32) * 2) - 1
out_bad, = bad(din=data)
print("emulation without Buffer still works:", out_bad.shape)
print("Do not rely on that for RTL — the fast join input has nowhere to hold packets.")


emulation without Buffer still works: (100, 1, 8)
Do not rely on that for RTL — the fast join input has nowhere to hold packets.


## Fix: `Buffer` on the fast branch

`Buffer(num_packets=...)` is a pass-through for values; in hardware it is a
packet FIFO. Size `num_packets` for how many packets the fast side may need to
hold while the slow side catches up (at least the imbalance you expect; using
the full frame count you plan to stream is a safe upper bound for small demos).


In [3]:
class Reconverge_with_buffer(Directed_acyclic_graph):
    """Same diamond, with a Buffer on the fast edge into the Add."""

    def __init__(self, depth=6, num_lanes=4, dw=16):
        self.depth = depth
        self.num_lanes = num_lanes
        self.dw = dw
        super().__init__(module_name="Reconverge_with_buffer")

    def init_topology(self):
        self.input = self.create_input("din", self.dw, self.num_lanes, "BF16")
        self.output = self.create_output("dout", self.dw, self.num_lanes, "BF16")

        self.slow = self.add_node(
            Sequential(
                name="Slow_path",
                layers=[
                    Accumulator(num_packets=100, dw_in=self.dw, num_lanes=self.num_lanes, dtype="BF16"),
                    Repeater(repeat=100),
                ]
            )
        )

        # Hold packets on the fast path so the fork is not stalled by the join
        self.fast_buf = self.add_node(
            Buffer(
                num_packets=100,
                num_lanes=self.num_lanes,
                dw=self.dw,
                dtype="BF16",
            )
        )
        self.join = self.add_node(Add(num_lanes=self.num_lanes, dw=self.dw, dtype="BF16"))

        self.connect(self.input, self.slow.inputs.din)
        self.connect(self.input, self.fast_buf.inputs.din)

        self.connect(self.slow.outputs.dout, self.join.inputs.dina)
        self.connect(self.fast_buf.outputs.dout, self.join.inputs.dinb)
        self.connect(self.join.outputs.dout, self.output)


good = Reconverge_with_buffer()
out_good, = good(din=data)
print("with Buffer:", out_good.shape)


with Buffer: (100, 1, 8)


## Checklist for hand-built DAGs

1. Find every **fan-out** (one port connected to multiple destinations).
2. Find every **join** (`Add`, `Multiply`, …) where those branches meet again.
3. If one branch is much deeper / higher latency, put a `Buffer` on the **faster**
   edge into the join, with `num_packets` large enough for the packets that must
   wait.
4. Prove it in **RTL simulation** with several packets back-to-back — single-packet
   runs often hide the hang.

Real models (e.g. Magika under `external/spatial_examples/magika/`) insert
`Buffer(num_packets=...)` on reconverging paths for the same reason.


## Optional: simulate both graphs to see the hang

Simulation should finish within 1 minute.


In [4]:
# ret = run_flow(bad, sim=True) # Should not load all 100 input packets.
ret = run_flow(good, sim=True)
print("buffered reconverge sim exit:", ret)


Dumping input and output data...
Running model RECONVERGE_WITH_BUFFER

Running Simulation (vcs)...
Running VCS in sim_RECONVERGE_WITH_BUFFER (device_family=Agilex 7)
*** Using c compiler gcc instead of cc ...
                         Chronologic VCS (TM)
      Version V-2023.12-SP2-1_Full64 -- Fri Sep  4 12:32:07 2026

                    Copyright (c) 1991 - 2024 Synopsys, Inc.
   This software and the associated documentation are proprietary to Synopsys,
 Inc. This software may only be used in accordance with the terms and conditions
 of a written license agreement with Synopsys, Inc. All other use, reproduction,
   or distribution of this software is strictly prohibited.  Licensed Products
     communicate with Synopsys servers for the purpose of providing software
    updates, detecting software piracy and verifying that customers are using
    Licensed Products in conformity with the applicable License Key for such
  Licensed Products. Synopsys will use information gathered in con

/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: warning: rmar_nd.o: missing .note.GNU-stack section implies executable stack
/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: NOTE: This behaviour is deprecated and will be removed in a future version of the linker


rm -f _cuarc*.so _csrc*.so pre_vcsobj_*.so share_vcsobj_*.so
if [ -x ../simv ]; then chmod a-x ../simv; fi
g++  -o ../simv   -no-pie    -rdynamic  -Wl,-rpath='$ORIGIN'/simv.daidir -Wl,-rpath=./simv.daidir -Wl,-rpath=/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib -L/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib  -Wl,-rpath-link=./   objs/amcQw_d.o   _4806_archive_1.so   SIM_l.o       rmapats_mop.o rmapats.o rmar.o rmar_nd.o  rmar_llvm_0_1.o rmar_llvm_0_0.o            -lnuma -lvirsim -lerrorinf -lsnpsmalloc -lvfs    -lvcsnew -lsimprofile -lreader_common /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/libBA.a -luclinative /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_tls.o   -Wl,-whole-archive  -lvcsucli    -Wl,-no-whole-archive       ./../simv.daidir/vc_hdrs.o    /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_save_restore_new.o -ldl  -lc -lm -lpthread -ldl 
/nfs/site/disks/psg_ctools_1/